# QRCL Memory Benchmark with Static Phase Correction

This notebook benchmarks how well a quantum state (|+⟩) is preserved over repeated QRCL relay cycles,
comparing runs with and without static phase correction under simulated noise.


In [ ]:
from qiskit import QuantumCircuit, Aer, transpile, execute
from qiskit.providers.aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error
import numpy as np
import matplotlib.pyplot as plt

## Define Noise Model

In [ ]:
noise_model = NoiseModel()
gate_error_prob = 0.01
t1 = 50e3  # ns
t2 = 70e3
gate_time = 200

cnot_error = depolarizing_error(gate_error_prob, 2)
rz_error = thermal_relaxation_error(t1, t2, gate_time)

noise_model.add_all_qubit_quantum_error(cnot_error, ['cx'])
noise_model.add_all_qubit_quantum_error(rz_error, ['rz'])

backend = Aer.get_backend('aer_simulator')

## Define QRCL Memory Test Function

In [ ]:
def qrcl_memory_test(num_cycles=20, shots=1024, apply_phase_correction=True):
    phi0 = np.pi / 16
    freq = 1 / 4
    delta_t = 1
    zero_probs = []

    for n in range(1, num_cycles + 1):
        qc = QuantumCircuit(2, 1)
        qc.h(0)

        for i in range(n):
            phi_n = phi0 * np.sin(2 * np.pi * freq * i * delta_t)
            qc.cx(0, 1)
            qc.cx(1, 0)
            qc.cx(0, 1)
            if apply_phase_correction:
                qc.rz(-phi_n, 1)
            qc.cx(0, 1)
            qc.cx(1, 0)
            qc.cx(0, 1)
            if apply_phase_correction:
                qc.rz(-phi_n, 0)

        qc.h(0)
        qc.measure(0, 0)

        transpiled = transpile(qc, backend)
        result = execute(transpiled, backend, noise_model=noise_model, shots=shots).result()
        counts = result.get_counts()
        zero_prob = counts.get('0', 0) / shots
        zero_probs.append(zero_prob)

    return zero_probs

## Run and Plot Memory Benchmark

In [ ]:
corrected = qrcl_memory_test(apply_phase_correction=True)
uncorrected = qrcl_memory_test(apply_phase_correction=False)

plt.plot(corrected, label='With Phase Correction', marker='o')
plt.plot(uncorrected, label='Without Correction', marker='x')
plt.title("QRCL Memory Benchmark: Probability of Measuring |0⟩")
plt.xlabel("Relay Cycles")
plt.ylabel("P(|0⟩) after H")
plt.ylim(0.4, 1.01)
plt.grid(True)
plt.legend()
plt.show()